## Preprocessing before Modeling_Deep Neural Networks(DNN)

#### 8.1. gradient 기울기 소실 문제
- 심층 신경망에서 gradient가 전달되다가 점차 0에 가까워짐
- 예를 들어, 시그모이드 활성화 함수 특성상 아주 큰 값이나 작은 값이 들어오면 미분값이 거의 0이 됌
- 이를 해결하기 위해 심층 신경망에서 ReLu함수를 활용
- @https://playground.tensorflow.org

In [ ]:
import numpy as np
# import tensorflow as tf, keras
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Normalization 

#### 8.2. 손실함수 선택 문제와 softmax 활성화 함수

- softmax 함수 : $\frac{e^{2.0}}{e^{2.0}+e^{1.0}+e^{0.1}}$
  - 0일 확률 0.1, 1일 확률 0.7, 0일 확률 0.2
  - 큰 값은 더 커지고, 작은 값은 더 작아지는 특성
  - 정보이론에서의 불확실성 척도
  - 손실확률 분포 : Cross Entropy = $H(p, q) = -\Sigma_{x}p(x)log_{n}{q(x)}$
  
- maximum likelihood estimation : $p(d|x) = p(d=1|x)^{d} \cdots p(d=0|x)^{1-d}$
  - 주어진 데이터를 바르게 예측할 확률을 최대화한 것
  
- cross entropy loss [=negative log likelihood]
  - 다중 클래스 분류 문제에서 모델의 출력을 softmax 함수를 통해 확률값으로 변환 후, 실제 레이블과의 차이를 최소화하기 위해 사용
  - 곱셈을 반복하면 underflow가 생기므로 log로 변환
  - 이진 교차 엔트로피(BCE) 
  - categorical cross entropy

In [ ]:
# binary Cross Entropy [이진분류]

y_true = [ [1], [0], [0], [1] ]
# 정답 (0 또는 1), shape = (샘플 수, 1)
y_pred = [ [0.8], [0.3], [0.5], [0.9] ]
# 모델이 예측한 "클래스 1일 확률" (0~1 사이), shape = (샘플 수, 1)

binaryCrossEntropy = tf.keras.losses.BinaryCrossentropy()
# BinaryCrossentropy() 객체 생성
print(
    binaryCrossEntropy(y_true, y_pred).numpy()
)

0.34458154


In [ ]:
# categorical Cross Entropy [다중분류, 정답이 one-hot]

y_true = [ [0.0, 1.0, 0.0], [0.0, 0.0, 1.0], [1.0, 0.0, 0.0] ]
# one-hot 인코딩된 정답, shape = (샘플 수, 클래스 수)
y_pred = [ [0.6, 0.3, 0.1], [0.3, 0.6, 0.1], [0.1, 0.7, 0.2] ]
# 클래스별 확률 (softmax 출력), 각 행의 합이 1

categoricalCrossEntropy = tf.keras.losses.CategoricalCrossentropy()
print(
    categoricalCrossEntropy(y_true, y_pred).numpy()
)

1.936381


In [ ]:
# sparse Cross Entropy [정답 레이블 정수, [0, 클래스 수-1] 범위]

y_true = np.array([1, 2, 0]) # dtype = np.int32
# one-hot이 아니라 "정답 클래스 번호"(정수), shape = (샘플 수,)
y_pred = np.array([ [0.6, 0.3, 0.1], [0.3, 0.6, 0.1], [0.1, 0.7, 0.2] ]) # dtype = np.int32 
# CCE와 동일하게 클래스별 확률

sparseCrossEntropy = tf.keras.losses.SparseCategoricalCrossentropy()
print(
    sparseCrossEntropy(y_true, y_pred).numpy()
) # raise ValueError("Structures don't have the same nested structure.")

1.936381


- cross entropy의 계산
  - 교차 엔트로피: 실제 정답 데이터의 확률 분포와 모델이 예측한 확률 분포(\(q_{i}\))가 얼마나 다른지(오차)를 측정하는 손실 함수
  - $E = -\sum_{i} t_{i} \log y_{i} \\$ ($t$: 목표 출력, $y$: 실제 출력, $\log$: 자연로그) 

    | 입력 샘플 | 실제 출력 1 | 실제 출력 2 | 실제 출력 3 | 목표 출력 1 | 목표 출력 2 | 목표 출력 3 |
    |:---:|:---:|:---:|:---:|:---:|:---:|:---:|
    | 샘플 #1 | 0.1 | 0.3 | 0.6 | 0 | 0 | 1 |
    | 샘플 #2 | 0.2 | 0.6 | 0.2 | 0 | 1 | 0 |
    | 샘플 #3 | 0.3 | 0.4 | 0.3 | 1 | 0 | 0 |

- 첫 번째 샘플의 교차 엔트로피
  - $-(\log(0.1)\cdot 0 + \log(0.3)\cdot 0 + \log(0.6)\cdot 1) = -(0 + 0 - 0.51) = 0.51$
- 두 번째 샘플의 교차 엔트로피
  - $-(\log(0.2)\cdot 0 + \log(0.6)\cdot 1 + \log(0.2)\cdot 0) = -(0 - 0.51 + 0) = 0.51
$
- 세 번째 샘플의 교차 엔트로피
  - $-(\log(0.3)\cdot 1 + \log(0.4)\cdot 0 + \log(0.3)\cdot 0) = -(-1.20 + 0 + 0) = 1.20$
- 3가지 샘플의 평균 교차 엔트로피 오류는
  - $\frac{0.51 + 0.51 + 1.20}{3} = 0.74$

---

#### 8.3. 가중치 초기화 문제

- breaking the symmetry
- 난수 가중치 초기값 : $var(W_{i}) = \frac{2}{N_{in} + N_{out}}$
- 범주형 변수 encoding (3) : integer encoding, one hot encoding, embedding

In [13]:
X = np.array( [["Korea", 44, 7200], ["Japan", 27, 4800], ["China", 30, 6100]] )
print(X)

[['Korea' '44' '7200']
 ['Japan' '27' '4800']
 ['China' '30' '6100']]


In [14]:
# 정수 encoding

X_integer_encoding = X.copy()
# 원본 X를 건드리지 않도록 복사본(copy)에서 작업

labelEncoder = LabelEncoder()
# fit_transform: 고유한 값을 사전순으로 정렬하여 0, 1, 2... 문자열 레이블 부여한 후 변환하는 인코더 객체 생성
X_integer_encoding[:, 0] = labelEncoder.fit_transform(X_integer_encoding[:, 0])
print(X_integer_encoding)

[['2' '44' '7200']
 ['1' '27' '4800']
 ['0' '30' '6100']]


In [ ]:
# one-hot encoding

X_onehot_encoding = X.copy()
# 원본 X를 건드리지 않도록 복사본(copy)에서 작업

oneHotEncoder = OneHotEncoder()
# tf.keras.utils.to_categorical()

XX = oneHotEncoder.fit_transform(
    X_onehot_encoding[:, 0].reshape(-1, 1)).toarray()
# 원하는 열을 뽑아서 2차원 배열로 만들어서 전달
# 정렬 순서는 China = 0, Japan = 1, Korea = 2
print(XX)

X_onehot_encoding = np.delete(X_onehot_encoding, [0], axis = 1)
# 0번째 열(국가 문자열) 열방향 삭제
X_onehot_encoding = np.concatenate((XX, X_onehot_encoding), axis = 1)
# 원-핫 열과 나머지 열을 가로로 병합
print(X_onehot_encoding)

[[0. 0. 1.]
 [0. 1. 0.]
 [1. 0. 0.]]
[['0.0' '0.0' '1.0' '44' '7200']
 ['0.0' '1.0' '0.0' '27' '4800']
 ['1.0' '0.0' '0.0' '30' '6100']]


#### 8.4. 데이터 스케일링 방법

1. 정규화 (Min-Max Scaling)
   - 데이터를 $[0, 1]$ 범위로 변환
   - $x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$
2. 표준화 (Z-score Standardization)
   - 평균이 $0$, 표준편차가 $1$이 되도록 변환
   - $z = \frac{x - \mu}{\sigma}$

---

#### 8.5. 과잉 적합과 과소 적합

- 영화 리뷰를 분류하는 문제
  - IMDB 사이트의 영화리뷰 게시판 속 리뷰가 긍정적인지 부정적인지 파악하는 신경망